<a href="https://colab.research.google.com/github/Alessandro-json/AI_PostProcessing_Detection/blob/main/notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project 2 - Joint Detection of AI-Generated Images and Post-Processing Alterations in Real-World Scenarios

# Imports
This section imports the libraries used throughout the notebook.

In [ ]:
import os
import json
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image


# Globals

In this section we define the main paths and global variables used by the notebook.

Keeping these variables in one place makes the notebook easier to modify and reduces the risk of inconsistent paths across different experiments.

In [ ]:
# Choose where the project will be stored in Colab.
WORKDIR = Path('/content')

REPO_URL = 'https://github.com/Alessandro-json/AI_PostProcessing_Detection'

# Repository folder name after git clone.
REPO_DIR = WORKDIR / 'REPO'

# Main paths used by the scripts.
TRAIN_CSV = 'data/splits/train_balanced.csv'
VAL_CSV = 'data/splits/val_balanced.csv'
TEST_CSV = 'data/splits/test_balanced.csv'

IMAGE_ROOT = "/content/data/raw/RRDataset_subset"
CHECKPOINT_NAME = 'best_rgb.pt'
CHECKPOINT_PATH = f'checkpoints/{CHECKPOINT_NAME}'
DEPTH_ROOT = "/content/drive/MyDrive/CV_Project/depth_maps"

DATASET_FILE_ID = "1HJamsAB4Lj90xNjA6tkfIrstHGJqklsI"
DATASET_ZIP_PATH = "/content/RRDataset_subset.zip"

# Training hyperparameters for the first baseline.
EPOCHS = 10
BATCH_SIZE = 32
IMAGE_SIZE = 224
NUM_WORKERS = 2
LEARNING_RATE = 1e-4

# Multi-task loss weights.
LAMBDA_FAKE = 1.0
LAMBDA_TRANSFORM = 1.0


# Repository setup

The project code is stored in a GitHub repository and imported inside Colab.  
If the repository already exists in the working directory, we update it with `git pull`; otherwise, we clone it from GitHub.


In [ ]:
%cd {WORKDIR}

if REPO_DIR.exists():
    %cd {REPO_DIR}
    !git pull
else:
    !git clone {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}


# Install dependencies

The required Python packages are installed from `requirements.txt`.

In [ ]:
!pip install -q -r requirements.txt

# Utils

This section defines helper functions used to inspect the dataset, verify paths, load evaluation metrics, and visualize saved results.

In [ ]:
def show_csv_summary(csv_path):
    """Print a quick summary of one split CSV."""
    path = Path(csv_path)
    if not path.exists():
        print(f'Missing file: {path}')
        return None

    df = pd.read_csv(path)
    print(f'File: {path}')
    print(f'Rows: {len(df)}')
    print('Columns:', list(df.columns))

    if 'fake_label' in df.columns:
        print('\nFake label distribution:')
        print(df['fake_label'].value_counts().sort_index())

    if 'transform_label' in df.columns:
        print('\nTransform label distribution:')
        print(df['transform_label'].value_counts().sort_index())

    if {'fake_label', 'transform_label'}.issubset(df.columns):
        print('\nJoint distribution:')
        print(pd.crosstab(df['transform_label'], df['fake_label'], rownames=['transform'], colnames=['fake']))

    return df


In [ ]:
def show_image_exists_check(df, image_root, n=5):
    """Check whether the first n image paths exist."""
    if df is None:
        return

    root = Path(image_root)
    print(f'Image root: {root}')

    for rel_path in df['image_path'].head(n):
        full_path = root / rel_path
        print(full_path, 'OK' if full_path.exists() else 'MISSING')

In [ ]:
def show_evaluation_outputs(output_dir):
    """
    Show evaluation metrics and confusion matrices saved by evaluate_RGB.py.
    """

    output_dir = Path(output_dir)
    metrics_path = output_dir / "metrics.json"

    if not metrics_path.exists():
        print(f"Metrics file not found: {metrics_path}")
        return

    # Load metrics saved by evaluate_RGB.py.
    with open(metrics_path, "r", encoding="utf-8") as f:
        metrics = json.load(f)

    # Convert metrics dictionary into a readable table.
    rows = []

    for metric_name, value in metrics.items():
        if isinstance(value, dict):
            for sub_metric_name, sub_value in value.items():
                rows.append({
                    "Metric": metric_name,
                    "Group": sub_metric_name,
                    "Value": sub_value,
                })
        else:
            rows.append({
                "Metric": metric_name,
                "Group": "-",
                "Value": value,
            })

    metrics_df = pd.DataFrame(rows)

    # Format numeric values to 4 decimals.
    if not metrics_df.empty:
        metrics_df["Value"] = metrics_df["Value"].apply(
            lambda x: f"{x:.4f}" if isinstance(x, (int, float)) else x
        )

    print("Evaluation metrics")
    display(metrics_df)

    # Show saved confusion matrices as images.
    fake_cm_path = output_dir / "confusion_fake.png"
    transform_cm_path = output_dir / "confusion_transform.png"

    if fake_cm_path.exists():
        print("\nReal/Fake confusion matrix")
        display(Image(filename=str(fake_cm_path)))

    if transform_cm_path.exists():
        print("\nTransformation confusion matrix")
        display(Image(filename=str(transform_cm_path)))

    if not fake_cm_path.exists() and not transform_cm_path.exists():
        print("\nNo confusion matrix images found in this folder.")

In [ ]:
def build_comparison_df(results_dict):
    rows = []

    for model_name, metrics_path in results_dict.items():
        metrics_path = Path(metrics_path)

        if not metrics_path.exists():
            print(f"Missing file: {metrics_path}")
            continue

        with open(metrics_path, "r") as f:
            metrics = json.load(f)

        rows.append({
            "model": model_name,
            "fake_accuracy": metrics.get("fake_accuracy"),
            "fake_f1_macro": metrics.get("fake_f1_macro"),
            "transform_accuracy": metrics.get("transform_accuracy"),
            "transform_f1_macro": metrics.get("transform_f1_macro"),
            "fake_acc_original": metrics.get("fake_accuracy_by_transform", {}).get("original"),
            "fake_acc_transfer": metrics.get("fake_accuracy_by_transform", {}).get("transfer"),
            "fake_acc_redigital": metrics.get("fake_accuracy_by_transform", {}).get("redigital"),
        })

    comparison_df = pd.DataFrame(rows)

    if len(comparison_df) > 0:
        comparison_df = comparison_df.sort_values(
            by=["fake_accuracy", "transform_accuracy"],
            ascending=False,
        )

    return comparison_df


def plot_global_accuracy(comparison_df, title):
    plot_df = comparison_df.set_index("model")

    plot_df[["fake_accuracy", "transform_accuracy"]].plot(
        kind="bar",
        figsize=(12, 5),
    )

    plt.title(title)
    plt.ylabel("Accuracy")
    plt.ylim(0, 1)
    plt.xticks(rotation=45, ha="right")
    plt.grid(axis="y")
    plt.tight_layout()
    plt.show()


def plot_fake_accuracy_by_transformation(comparison_df, title):
    plot_df = comparison_df.set_index("model")

    plot_df[[
        "fake_acc_original",
        "fake_acc_transfer",
        "fake_acc_redigital",
    ]].plot(
        kind="bar",
        figsize=(12, 5),
    )

    plt.title(title)
    plt.ylabel("Accuracy")
    plt.ylim(0, 1)
    plt.xticks(rotation=45, ha="right")
    plt.grid(axis="y")
    plt.tight_layout()
    plt.show()

In [ ]:
def load_results(results_dict):
    rows = []
    for name, path in results_dict.items():
        p = Path(path)
        if not p.exists():
            print(f"[SKIP] {path}")
            continue
        with open(p) as f:
            m = json.load(f)
        by_t = m.get("fake_accuracy_by_transform", {})
        rows.append({
            "model":              name,
            "fake_accuracy":      m.get("fake_accuracy"),
            "transform_accuracy": m.get("transform_accuracy"),
            "fake_f1_macro":      m.get("fake_f1_macro"),
            "transform_f1_macro": m.get("transform_f1_macro"),
            "fake_acc_original":  by_t.get("original"),
            "fake_acc_transfer":  by_t.get("transfer"),
            "fake_acc_redigital": by_t.get("redigital"),
        })
    return pd.DataFrame(rows)


def compare_resnet_vs_vit(resnet_results: dict, vit_results: dict):
    resnet_df = load_results(resnet_results)
    vit_df    = load_results(vit_results)
    all_df    = pd.concat([resnet_df, vit_df], ignore_index=True)

    display_df = all_df.copy()
    for col in ["fake_accuracy", "transform_accuracy", "fake_f1_macro", "transform_f1_macro"]:
        if col in display_df.columns:
            display_df[col] = display_df[col].apply(
                lambda x: f"{x:.4f}" if pd.notna(x) else "-"
            )

    print("=== Comparative Table ResNet vs ViT ===")
    if display_df.empty:
        print("No results available yet.")
    else:
        display(display_df[["model", "fake_accuracy", "transform_accuracy",
                             "fake_f1_macro", "transform_f1_macro"]])

    fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)

    for ax, df, title, color_fake, color_transform in zip(
        axes,
        [resnet_df, vit_df],
        ["ResNet18 models", "ViT-Small models"],
        ["steelblue", "mediumpurple"],
        ["coral",     "tomato"],
    ):
        if df.empty:
            ax.set_title(f"{title} — No results")
            continue

        x     = range(len(df))
        width = 0.35

        ax.bar([i - width/2 for i in x], df["fake_accuracy"],
               width=width, label="Fake accuracy", color=color_fake, alpha=0.85)
        ax.bar([i + width/2 for i in x], df["transform_accuracy"],
               width=width, label="Transform accuracy", color=color_transform, alpha=0.85)

        ax.set_xticks(list(x))
        ax.set_xticklabels(df["model"], rotation=35, ha="right", fontsize=8)
        ax.set_ylim(0, 1.08)
        ax.set_ylabel("Accuracy")
        ax.set_title(title, fontsize=12)
        ax.legend(loc="lower right")
        ax.grid(axis="y", alpha=0.35)
        ax.axhline(y=0.5, color="gray", linestyle="--", linewidth=0.8, alpha=0.6)

    fig.suptitle("ResNet18 vs ViT-Small — Global Accuracy", fontsize=14, y=1.01)
    plt.tight_layout()
    Path("results").mkdir(exist_ok=True)
    plt.savefig("results/resnet_vs_vit_accuracy.png", dpi=200, bbox_inches="tight")
    plt.show()

    transform_cols = ["fake_acc_original", "fake_acc_transfer", "fake_acc_redigital"]
    labels         = ["Original", "Transfer", "Re-digital"]
    colors         = ["#4c72b0", "#55a868", "#c44e52"]

    fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)

    for ax, df, title in zip(
        axes,
        [resnet_df, vit_df],
        ["ResNet18 — Fake Acc by Transform", "ViT-Small — Fake Acc by Transform"],
    ):
        if df.empty:
            ax.set_title(f"{title} — No results")
            continue

        sub = df.dropna(subset=transform_cols, how="all")
        if sub.empty:
            ax.set_title(f"{title} — No results")
            continue

        x     = range(len(sub))
        width = 0.25

        for k, (col, label, color) in enumerate(zip(transform_cols, labels, colors)):
            ax.bar(
                [i + (k - 1) * width for i in x],
                sub[col].fillna(0),
                width=width, label=label, color=color, alpha=0.85,
            )

        ax.set_xticks(list(x))
        ax.set_xticklabels(sub["model"], rotation=35, ha="right", fontsize=8)
        ax.set_ylim(0, 1.08)
        ax.set_ylabel("Fake Accuracy")
        ax.set_title(title, fontsize=11)
        ax.legend(loc="lower right")
        ax.grid(axis="y", alpha=0.35)
        ax.axhline(y=0.5, color="gray", linestyle="--", linewidth=0.8, alpha=0.6)

    fig.suptitle("ResNet18 vs ViT-Small — Fake Accuracy by Transformation Type",
                 fontsize=13, y=1.01)
    plt.tight_layout()
    plt.savefig("results/resnet_vs_vit_by_transform.png", dpi=200, bbox_inches="tight")
    plt.show()

    print("\n=== Best model per group ===")
    for name, df in [("ResNet18", resnet_df), ("ViT-Small", vit_df)]:
        if df.empty:
            print(f"{name}: no results available")
            continue
        valid = df.dropna(subset=["fake_accuracy"])
        if valid.empty:
            print(f"{name}: no results available")
            continue
        best = valid.loc[valid["fake_accuracy"].idxmax()]
        print(f"{name}: {best['model']}"
              f"  →  fake={best['fake_accuracy']:.4f}"
              f"  transform={best['transform_accuracy']:.4f}")

# Data preparation and inspection

Before training any model, we inspect the dataset splits and verify that the image paths are correctly resolved.

The project uses a balanced subset of RRDataset, containing both real and AI-generated images across three transformation categories:

- original images;
- internet-transmitted images;
- re-digitized images.



In [ ]:
train_df = show_csv_summary(TRAIN_CSV)

In [ ]:
val_df = show_csv_summary(VAL_CSV)

In [ ]:
test_df = show_csv_summary(TEST_CSV)

The dataset is downloaded only if the zip file is not already available and extracted only if the target image folder does not already exist.

In [ ]:
# Download the dataset only if the zip file is not already available.
if not Path(DATASET_ZIP_PATH).exists():
    print("Dataset zip not found. Downloading it with gdown...")
    os.system(f'gdown --id "{DATASET_FILE_ID}" -O "{DATASET_ZIP_PATH}"')
else:
    print(f"Dataset zip already exists: {DATASET_ZIP_PATH}")

In [ ]:
# Extract the dataset only if the extracted folder is not already available.
IMAGE_ROOT = Path(IMAGE_ROOT)
if not IMAGE_ROOT.exists():
    print("Extracted dataset folder not found. Extracting...")
    os.system(f'mkdir -p "{IMAGE_ROOT.parent}"')
    os.system(f'unzip -q "{DATASET_ZIP_PATH}" -d "{IMAGE_ROOT.parent}"')
else:
    print(f"Dataset already extracted: {IMAGE_ROOT}")

We verify that the image paths listed in the CSV files actually exist inside the dataset folder: if paths are wrong, training would fail later during data loading.

In [ ]:
show_image_exists_check(train_df, IMAGE_ROOT, n=5)

# RGB baselines

The first group of experiments uses only the RGB image as input.

We train and evaluate:

1. a **single-task real/fake baseline**;
2. a **single-task transformation baseline**;
3. several **RGB multi-task baselines** with different loss weightings.

This allows us to understand whether joint learning improves, hurts, or leaves unchanged the performance of the two tasks.

## Single-task Real/Fake baseline

This baseline trains the RGB model only on the binary real/fake task.

Its purpose is to measure how well a standard RGB model can distinguish real images from AI-generated images without receiving any supervision about post-processing transformations.

In [ ]:
# Train the real/fake single-task baseline.
!python src/train_RGB.py \
  --task fake \
  --train_csv {TRAIN_CSV} \
  --val_csv {VAL_CSV} \
  --image_root {IMAGE_ROOT} \
  --epochs {EPOCHS} \
  --batch_size {BATCH_SIZE} \
  --image_size {IMAGE_SIZE} \
  --num_workers {NUM_WORKERS} \
  --checkpoint_name best_rgb_fake.pt

We evaluate the fake-only model on the test split and report global real/fake metrics.

In [ ]:
# Evaluate the real/fake single-task baseline.
!python src/evaluate_RGB.py \
  --task fake \
  --csv_path {TEST_CSV} \
  --image_root {IMAGE_ROOT} \
  --checkpoint checkpoints/best_rgb_fake.pt \
  --output_dir results/rgb_fake \
  --batch_size {BATCH_SIZE} \
  --image_size {IMAGE_SIZE} \
  --num_workers {NUM_WORKERS}

The following outputs summarize the fake-only baseline:

- global real/fake accuracy;
- macro F1-score;
- real/fake accuracy by transformation type;
- real/fake confusion matrix.

In [ ]:
show_evaluation_outputs("results/rgb_fake")

## Single-task Transformation baseline

This baseline trains the RGB model only on the transformation classification task.

The model predicts whether an image is original, internet-transmitted, or re-digitized.  
This experiment helps us measure how recognizable post-processing traces are when the model is not simultaneously optimized for real/fake detection.

In [ ]:
# Train the transformation single-task baseline.
!python src/train_RGB.py \
  --task transform \
  --train_csv {TRAIN_CSV} \
  --val_csv {VAL_CSV} \
  --image_root {IMAGE_ROOT} \
  --epochs {EPOCHS} \
  --batch_size {BATCH_SIZE} \
  --image_size {IMAGE_SIZE} \
  --num_workers {NUM_WORKERS} \
  --checkpoint_name best_rgb_transform.pt

We evaluate the transformation-only model on the test split and report transformation accuracy, macro F1-score, and the transformation confusion matrix.

In [ ]:
# Evaluate the transformation single-task baseline.
!python src/evaluate_RGB.py \
  --task transform \
  --csv_path {TEST_CSV} \
  --image_root {IMAGE_ROOT} \
  --checkpoint checkpoints/best_rgb_transform.pt \
  --output_dir results/rgb_transform \
  --batch_size {BATCH_SIZE} \
  --image_size {IMAGE_SIZE} \
  --num_workers {NUM_WORKERS}

In [ ]:
show_evaluation_outputs("results/rgb_transform")

## RGB Multi-task baseline - manual weights 1.0 / 1.0

This is the standard multi-task baseline.

The model uses a shared RGB backbone and two independent classification heads:

- one head predicts the real/fake label;
- one head predicts the transformation label.

The total loss is computed as:

**L = 1.0 × L_fake + 1.0 × L_transform**

This setting gives equal importance to both tasks.

In [ ]:
# Train the joint RGB multi-task baseline with weights 1 1.
!python src/train_RGB.py \
  --task multitask \
  --train_csv {TRAIN_CSV} \
  --val_csv {VAL_CSV} \
  --image_root {IMAGE_ROOT} \
  --epochs {EPOCHS} \
  --batch_size {BATCH_SIZE} \
  --image_size {IMAGE_SIZE} \
  --num_workers {NUM_WORKERS} \
  --lambda_fake 1.0 \
  --lambda_transform 1.0 \
  --checkpoint_name best_rgb_multitask_1_1.pt

In [ ]:
# Evaluate the joint RGB multi-task baseline with weights 1 1.
!python src/evaluate_RGB.py \
  --task multitask \
  --csv_path {TEST_CSV} \
  --image_root {IMAGE_ROOT} \
  --checkpoint checkpoints/best_rgb_multitask_1_1.pt \
  --output_dir results/rgb_multitask_1_1 \
  --batch_size {BATCH_SIZE} \
  --image_size {IMAGE_SIZE} \
  --num_workers {NUM_WORKERS}

In [ ]:
show_evaluation_outputs("results/rgb_multitask_1_1")

## RGB Multi-task baseline - manual weights 1.0 / 2.0

In this experiment, the transformation loss receives a higher weight than the real/fake loss:

**L = 1.0 × L_fake + 2.0 × L_transform**

The goal is to test whether forcing the model to focus more on transformation recognition also improves real/fake detection.

In [ ]:
# Train the joint RGB multi-task baseline with weights 1 2.
!python src/train_RGB.py \
  --task multitask \
  --train_csv {TRAIN_CSV} \
  --val_csv {VAL_CSV} \
  --image_root {IMAGE_ROOT} \
  --epochs {EPOCHS} \
  --batch_size {BATCH_SIZE} \
  --image_size {IMAGE_SIZE} \
  --num_workers {NUM_WORKERS} \
  --lambda_fake 1.0 \
  --lambda_transform 2.0 \
  --checkpoint_name best_rgb_multitask_1_2.pt

In [ ]:
# Evaluate the joint RGB multi-task baseline with weights 1 2.
!python src/evaluate_RGB.py \
  --task multitask \
  --csv_path {TEST_CSV} \
  --image_root {IMAGE_ROOT} \
  --checkpoint checkpoints/best_rgb_multitask_1_2.pt \
  --output_dir results/rgb_multitask_1_2 \
  --batch_size {BATCH_SIZE} \
  --image_size {IMAGE_SIZE} \
  --num_workers {NUM_WORKERS}

In [ ]:
show_evaluation_outputs("results/rgb_multitask_1_2")

## RGB Multi-task baseline - manual weights 2.0 / 1.0

In this experiment, the real/fake loss receives a higher weight than the transformation loss:

**L = 2.0 × L_fake + 1.0 × L_transform**

This setting prioritizes the main forensic task while still keeping transformation classification as an auxiliary task.

In [ ]:
# Train the joint RGB multi-task baseline with weights 2 1.
!python src/train_RGB.py \
  --task multitask \
  --train_csv {TRAIN_CSV} \
  --val_csv {VAL_CSV} \
  --image_root {IMAGE_ROOT} \
  --epochs {EPOCHS} \
  --batch_size {BATCH_SIZE} \
  --image_size {IMAGE_SIZE} \
  --num_workers {NUM_WORKERS} \
  --lambda_fake 2.0 \
  --lambda_transform 1.0 \
  --checkpoint_name best_rgb_multitask_2_1.pt

In [ ]:
# Evaluate the joint RGB multi-task baseline with weights 2 1.
!python src/evaluate_RGB.py \
  --task multitask \
  --csv_path {TEST_CSV} \
  --image_root {IMAGE_ROOT} \
  --checkpoint checkpoints/best_rgb_multitask_2_1.pt \
  --output_dir results/rgb_multitask_2_1 \
  --batch_size {BATCH_SIZE} \
  --image_size {IMAGE_SIZE} \
  --num_workers {NUM_WORKERS}

In [ ]:
show_evaluation_outputs("results/rgb_multitask_2_1")

## RGB Multi-task baseline - Learned Uncertainty Weighting

This experiment replaces manually selected loss weights with learned uncertainty-based weights.

Instead of fixing the relative importance of the two tasks before training, the model learns how much each task should contribute to the total loss.

This is useful because the two tasks may have different difficulty levels. If one task is noisier or harder, the learned weighting mechanism can reduce its dominance and make the multi-task optimization more balanced.

In [ ]:
# Train the joint RGB multi-task baseline with Learned Uncertainty Weighting
!python src/train_RGB.py \
  --task multitask \
  --loss_weighting learned \
  --train_csv {TRAIN_CSV} \
  --val_csv {VAL_CSV} \
  --image_root {IMAGE_ROOT} \
  --epochs {EPOCHS} \
  --batch_size {BATCH_SIZE} \
  --image_size {IMAGE_SIZE} \
  --num_workers {NUM_WORKERS} \
  --checkpoint_name best_rgb_multitask_learned_weights.pt

In [ ]:
# Evaluate the joint RGB multi-task baseline with Learned Uncertainty Weighting
!python src/evaluate_RGB.py \
  --task multitask \
  --csv_path {TEST_CSV} \
  --image_root {IMAGE_ROOT} \
  --checkpoint checkpoints/best_rgb_multitask_learned_weights.pt \
  --output_dir results/rgb_multitask_learned_weights \
  --batch_size {BATCH_SIZE} \
  --image_size {IMAGE_SIZE} \
  --num_workers {NUM_WORKERS}

In [ ]:
show_evaluation_outputs("results/rgb_multitask_learned_weights")

## Results Comparison and Ablation Study

This section compares the RGB baselines and the RGB multi-task models.

The objective is to understand whether the transformation task helps the real/fake detector learn more robust features, or whether the two objectives interfere with each other during training.

We also compare different loss-weighting strategies, since changing the relative importance of the two losses can affect the balance between real/fake accuracy and transformation accuracy.

Finally, we report real/fake accuracy by transformation type to check whether the model remains reliable on original, transmitted, and re-digitized images.

In [ ]:
results = {
    "RGB fake-only": "results/rgb_fake/metrics.json",
    "RGB transform-only": "results/rgb_transform/metrics.json",
    "RGB multitask 1-1": "results/rgb_multitask_1_1/metrics.json",
    "RGB multitask 1-2": "results/rgb_multitask_1_2/metrics.json",
    "RGB multitask 2-1": "results/rgb_multitask_2_1/metrics.json",
    "RGB multitask learned": "results/rgb_multitask_learned_weights/metrics.json",
}

rows = []

for model_name, metrics_path in results.items():
    metrics_path = Path(metrics_path)

    if not metrics_path.exists():
        print(f"Missing file: {metrics_path}")
        continue

    with open(metrics_path, "r") as f:
        metrics = json.load(f)

    rows.append({
        "model": model_name,
        "fake_accuracy": metrics.get("fake_accuracy"),
        "fake_f1_macro": metrics.get("fake_f1_macro"),
        "transform_accuracy": metrics.get("transform_accuracy"),
        "transform_f1_macro": metrics.get("transform_f1_macro"),
        "fake_acc_original": metrics.get("fake_accuracy_by_transform", {}).get("original"),
        "fake_acc_transfer": metrics.get("fake_accuracy_by_transform", {}).get("transfer"),
        "fake_acc_redigital": metrics.get("fake_accuracy_by_transform", {}).get("redigital"),
    })

comparison_df = pd.DataFrame(rows)
comparison_df.sort_values(
    by=["fake_accuracy", "transform_accuracy"],
    ascending=False
)

### Global accuracy comparison

The following plot compares the main accuracy scores across RGB experiments.

In [ ]:
# Global Accuracy Comparison

plot_df = comparison_df.set_index("model")

plot_df[["fake_accuracy", "transform_accuracy"]].plot(
    kind="bar",
    figsize=(12, 5),
)

plt.title("Global Accuracy Comparison")
plt.ylabel("Accuracy")
plt.ylim(0, 1)
plt.xticks(rotation=45, ha="right")
plt.grid(axis="y")
plt.tight_layout()
plt.show()

### Real/Fake accuracy by transformation type

This analysis breaks down real/fake detection accuracy across transformation categories.

This is important because the main challenge of the project is not only detecting AI-generated images in clean conditions, but also maintaining robustness after real-world post-processing.

In [ ]:
# Real/Fake Accuracy by Transformation Type
plot_df[[
    "fake_acc_original",
    "fake_acc_transfer",
    "fake_acc_redigital",
]].plot(
    kind="bar",
    figsize=(12, 5),
)

plt.title("Real/Fake Accuracy by Transformation Type")
plt.ylabel("Accuracy")
plt.ylim(0, 1)
plt.xticks(rotation=45, ha="right")
plt.grid(axis="y")
plt.tight_layout()
plt.show()

#Frequency

GANs generate periodic artifacts, invisibles in RGB, but that can be detect in frequency domain.
Is used FFT on the amplitude log-spectrum.



## Standard frequency-based multi-task training

This cell trains the multi-task model using the frequency branch with equal loss weights for the two objectives.  
The model jointly optimizes real/fake detection and transformation classification, using:

**L_total = 1.0 * L_fake + 1.0 * L_transform**

This configuration is used as the standard frequency multi-task baseline. It allows us to evaluate whether frequency-domain cues, such as compression traces, high-frequency artifacts, and post-processing residuals, can improve the joint detection of AI-generated images and real-world transformations.

**Train**

In [ ]:
# Frequency multi-task using weights 1.0 / 1.0
!python src/train_freq.py \
  --train_csv        {TRAIN_CSV} \
  --val_csv          {VAL_CSV} \
  --image_root       {IMAGE_ROOT} \
  --epochs           {EPOCHS} \
  --batch_size       {BATCH_SIZE} \
  --image_size       {IMAGE_SIZE} \
  --num_workers      {NUM_WORKERS} \
  --lambda_fake      1.0 \
  --lambda_transform 1.0 \
  --checkpoint_name  best_freq_multitask_1_1.pt

**Evaluate**

In [ ]:
!python src/evaluate_freq.py \
  --csv_path    {TEST_CSV} \
  --image_root  {IMAGE_ROOT} \
  --checkpoint  checkpoints/best_freq_multitask_1_1.pt \
  --output_dir  results/freq_multitask_1_1 \
  --batch_size  {BATCH_SIZE} \
  --image_size  {IMAGE_SIZE} \
  --num_workers {NUM_WORKERS}

**Results**

In [ ]:
show_evaluation_outputs("results/freq_multitask_1_1")

**Comparison**

In [ ]:
results = {
"RGB multitask 1-1":"results/results/rgb_multitask_1_1/metrics.json",
"Frequency 1-1":"results/freq_multitask_1_1/metrics.json",
}

rows = []

for model_name, metrics_path in results.items():
    metrics_path = Path(metrics_path)

    if not metrics_path.exists():
        print(f"Missing file: {metrics_path}")
        continue

    with open(metrics_path, "r") as f:
        metrics = json.load(f)

    rows.append({
        "model": model_name,
        "fake_accuracy": metrics.get("fake_accuracy"),
        "fake_f1_macro": metrics.get("fake_f1_macro"),
        "transform_accuracy": metrics.get("transform_accuracy"),
        "transform_f1_macro": metrics.get("transform_f1_macro"),
        "fake_acc_original": metrics.get("fake_accuracy_by_transform", {}).get("original"),
        "fake_acc_transfer": metrics.get("fake_accuracy_by_transform", {}).get("transfer"),
        "fake_acc_redigital": metrics.get("fake_accuracy_by_transform", {}).get("redigital"),
    })



## Frequency model improvement: learned weighting and cosine scheduling

This cell trains an improved version of the frequency multi-task model.  
Compared to the basic frequency baseline with fixed loss weights, this experiment adds two optimization improvements:

1. learned uncertainty weighting, which lets the model automatically balance the real/fake loss and the transformation loss;
2. cosine learning-rate scheduling with warm-up, which makes the learning rate increase gradually at the beginning and then decrease smoothly during training.


In [ ]:
!python src/FrequencyAugumented.py \
  --train_csv {TRAIN_CSV} \
  --val_csv {VAL_CSV} \
  --image_root {IMAGE_ROOT} \
  --epochs {EPOCHS} \
  --batch_size {BATCH_SIZE} \
  --num_workers {NUM_WORKERS} \
  --scheduler cosine \
  --warmup_epochs 2 \
  --loss_weighting learned \
  --checkpoint_name best_freq.pt

**Evaluation**

In [ ]:
!python src/evaluate_freq_agu.py \
  --csv_path   {TEST_CSV} \
  --image_root {IMAGE_ROOT} \
  --checkpoint checkpoints/best_freq.pt\
  --output_dir results/freq_aug \
  --batch_size 32 --num_workers {NUM_WORKERS}

**Results**

In [ ]:
show_evaluation_outputs("results/freq_aug")

**Comparison**

In [ ]:
!python src/compare_freq_agu.py

In [ ]:
df = pd.read_csv("results/comparison_table.csv")
display(df)
display(Image(filename="results/comparison_global_accuracy.png"))
display(Image(filename="results/comparison_by_transform.png"))

#DEPTH

After the RGB baselines, we introduce depth as an auxiliary geometric cue in the multi-task setting.

We then evaluate two variants: RGB + Depth and RGB + Depth + Edge.

The RGB + Depth model tests whether scene geometry and foreground/background structure help the two tasks, while the RGB + Depth + Edge model investigates whether inconsistencies between visual edges and depth discontinuities provide additional forensic information.

##Depth map generation
Since the dataset does not provide ground-truth depth, pseudo-depth maps are generated offline with MiDaS and paired with each RGB image to provide an additional geometric cue.

In [ ]:
!python src/generate_depth_map.py \
  --csv_paths {TRAIN_CSV} {VAL_CSV} {TEST_CSV} \
  --image_root {IMAGE_ROOT} \
  --depth_root {DEPTH_ROOT} \
  --model_type MiDaS_small

After first generation depth maps are on drive so we do not need to comute them again. If they are not on drive they are generated again. In every case the maps are loaded on local Colab memory to make loading faster

In [ ]:
from pathlib import Path

DRIVE_DEPTH_ROOT = Path("/content/drive/MyDrive/CV_Project/depth_maps")
LOCAL_DEPTH_ROOT = Path("/content/depth_maps")
DEPTH_ROOT = str(LOCAL_DEPTH_ROOT)

DRIVE_DEPTH_ROOT.mkdir(parents=True, exist_ok=True)
LOCAL_DEPTH_ROOT.mkdir(parents=True, exist_ok=True)

drive_depth_files = list(DRIVE_DEPTH_ROOT.rglob("*.npy"))

print(f"Depth maps found on Drive: {len(drive_depth_files)}")

if len(drive_depth_files) == 0:
    print("No depth maps found on Drive. Generating them once and saving directly to Drive...")

    !python src/generate_depth_map.py \
      --csv_paths {TRAIN_CSV} {VAL_CSV} {TEST_CSV} \
      --image_root {IMAGE_ROOT} \
      --depth_root "{DRIVE_DEPTH_ROOT}" \
      --model_type MiDaS_small

else:
    print("Depth maps already exist on Drive. No need to regenerate them.")

print("Copying depth maps from Drive to local /content...")
!rsync -a --info=progress2 "{DRIVE_DEPTH_ROOT}/" "{LOCAL_DEPTH_ROOT}/"

DEPTH_ROOT = str(LOCAL_DEPTH_ROOT)

print("Local DEPTH_ROOT:", DEPTH_ROOT)
print("Some local depth maps:")
!find "{DEPTH_ROOT}" -type f -name "*.npy" | head

##RGB + Depth

This model combines RGB images with precomputed MiDaS depth maps. RGB features provide visual appearance information, while depth features add geometric cues about scene structure and spatial consistency.

The fused representation is then used in a multi-task setting for both real/fake detection and transformation classification.

In [ ]:
!python src/train_depth.py \
  --train_csv {TRAIN_CSV} \
  --val_csv {VAL_CSV} \
  --image_root {IMAGE_ROOT} \
  --depth_root {DEPTH_ROOT} \
  --checkpoint_name best_depth_uncertainty.pt \
  --epochs {EPOCHS} \
  --batch_size {BATCH_SIZE} \
  --image_size {IMAGE_SIZE} \
  --num_workers {NUM_WORKERS} \
  --lr {LEARNING_RATE} \
  --use_uncertainty_weighting \
  --no_edge

##Evaluate RGB + Depth

In [ ]:
!python src/evaluate_depth.py \
  --task multitask \
  --csv_path {TEST_CSV} \
  --image_root {IMAGE_ROOT} \
  --depth_root {DEPTH_ROOT} \
  --checkpoint checkpoints/best_depth_uncertainty.pt \
  --output_dir results/depth_only \
  --batch_size {BATCH_SIZE} \
  --image_size {IMAGE_SIZE} \
  --num_workers {NUM_WORKERS} \
  --no_edge

In [ ]:
show_evaluation_outputs("results/depth_only")

Compared to the RGB fake baseline, the RGB + Depth model slightly improves real/fake detection, suggesting that depth information can provide useful geometric cues in more challenging post-processing conditions.

##RGB + Depth + Edge

This model extends RGB + Depth by adding an edge-consistency map as an additional input. The edge map is computed by comparing RGB image edges with depth-map edges, in order to highlight possible inconsistencies between visual contours and estimated geometry.


In [ ]:
!python src/train_depth.py \
  --train_csv {TRAIN_CSV} \
  --val_csv {VAL_CSV} \
  --image_root {IMAGE_ROOT} \
  --depth_root {DEPTH_ROOT} \
  --checkpoint_name best_depth_edge_uncertainty.pt \
  --epochs {EPOCHS} \
  --batch_size {BATCH_SIZE} \
  --image_size {IMAGE_SIZE} \
  --num_workers {NUM_WORKERS} \
  --lr {LEARNING_RATE} \
  --use_uncertainty_weighting

##Evaluate RGB+Depth+edge

In [ ]:
!python src/evaluate_depth.py \
  --task multitask \
  --csv_path {TEST_CSV} \
  --image_root {IMAGE_ROOT} \
  --depth_root {DEPTH_ROOT} \
  --checkpoint checkpoints/best_depth_edge_uncertainty.pt \
  --output_dir results/depth_edge \
  --batch_size {BATCH_SIZE} \
  --image_size {IMAGE_SIZE} \
  --num_workers {NUM_WORKERS}

In [ ]:
show_evaluation_outputs("results/depth_edge")

The edge-consistency branch introduces additional geometric information, but also increases model complexity. This leads to mild overfitting and a small decrease in real/fake validation accuracy, although transformation classification slightly improves.
We should use only depth without edge

##Comparison
We compare RGB + Depth with RGB + Depth + Edge

In [ ]:
depth_results = {
    "RGB + Depth": "results/depth_only/metrics.json",
    "RGB + Depth + Edge": "results/depth_edge/metrics.json",
}

depth_comparison_df = build_comparison_df(depth_results)

depth_comparison_df


plot_global_accuracy(
    depth_comparison_df,
    "Depth Ablation: Global Accuracy Comparison"
)



plot_fake_accuracy_by_transformation(
    depth_comparison_df,
    "Depth Ablation: Real/Fake Accuracy by Transformation Type"
)




#RGB + DEPTH + FREQUENCY

This model combines three complementary cues: RGB appearance, MiDaS pseudo-depth maps, and frequency-domain information computed from the RGB image.

Depth provides geometric information about scene structure, while frequency features can highlight spectral artifacts related to image generation, compression, transmission, or re-digitization. The three representations are fused and used in a multi-task setting for both real/fake detection and transformation classification.

In [ ]:
!python src/train_depth_frequency.py \
  --train_csv {TRAIN_CSV} \
  --val_csv {VAL_CSV} \
  --image_root {IMAGE_ROOT} \
  --depth_root {DEPTH_ROOT} \
  --checkpoint_name best_depth_frequency_uncertainty.pt \
  --checkpoint_dir checkpoints \
  --epochs {EPOCHS} \
  --batch_size {BATCH_SIZE} \
  --image_size {IMAGE_SIZE} \
  --num_workers {NUM_WORKERS} \
  --lr {LEARNING_RATE} \
  --use_uncertainty_weighting

##Evaluate RGB + DEPTH + FREQUENCY

In [ ]:
!python src/evaluation_depth_frequency.py \
  --csv_path {TEST_CSV} \
  --image_root {IMAGE_ROOT} \
  --depth_root {DEPTH_ROOT} \
  --checkpoint checkpoints/best_depth_frequency_uncertainty.pt \
  --output_dir results/depth_frequency_uncertainty \
  --batch_size {BATCH_SIZE} \
  --image_size  {IMAGE_SIZE} \
  --num_workers {NUM_WORKERS}

In [ ]:
show_evaluation_outputs("results/depth_frequency_uncertainty")

Adding frequency module to depth module improves transformation classification. This suggests that spectral cues are useful for detecting post-processing traces, although they do not provide a consistent benefit for real/fake detection.

#RGB + Depth + Frequency gated

This model introduces a gated fusion mechanism on top of the RGB + Depth + Frequency architecture.

The gate learns how much weight to assign to RGB, depth, and frequency separately for each task. This allows the fake detection head and the transformation classification head to use different modality combinations.

This experiment tests whether task-specific fusion improves performance compared to the shared non-gated multimodal representation.

In [ ]:
DRIVE_CHECKPOINT_DIR = "/content/drive/MyDrive/CV_Project/checkpoints"

In [ ]:
!python src/train_depth_frequency_gated.py \
  --train_csv {TRAIN_CSV} \
  --val_csv {VAL_CSV} \
  --image_root {IMAGE_ROOT} \
  --depth_root {DEPTH_ROOT} \
  --checkpoint_name best_depth_frequency_gated_uncertainty.pt \
  --checkpoint_dir checkpoints \
  --epochs 10 \
  --batch_size {BATCH_SIZE} \
  --image_size {IMAGE_SIZE} \
  --num_workers {NUM_WORKERS} \
  --lr {LEARNING_RATE} \
  --use_uncertainty_weighting

##Evaluation RGB + DEPTH + FREQUENCY GATED

In [ ]:
!python src/evaluate_depth_frequency_gated.py \
  --csv_path {TEST_CSV} \
  --image_root {IMAGE_ROOT} \
  --depth_root {DEPTH_ROOT} \
  --checkpoint checkpoints/best_depth_frequency_gated_uncertainty.pt \
  --output_dir results/depth_frequency_gated_uncertainty \
  --batch_size {BATCH_SIZE} \
  --image_size  {IMAGE_SIZE} \
  --num_workers {NUM_WORKERS}


In [ ]:
show_evaluation_outputs("results/depth_frequency_gated_uncertainty")

Gated fusion does not improve the overall performance: fake accuracy slightly increases, but transformation accuracy decreases, leading to a lower final score than the non-gated model.

#Comparison
We do the comparison betweeen RGB + Depth + Frequency and RGB + Depth + Frequency Gated

In [ ]:
depth_frequency_results = {
    "RGB+Depth+Frequency": "results/depth_frequency_uncertainty/metrics.json",
    "RGB+Depth+Frequency Gated": "results/depth_frequency_gated_uncertainty/metrics.json",
}

depth_frequency_comparison_df = build_comparison_df(depth_frequency_results)

display(depth_frequency_comparison_df)

plot_global_accuracy(
    depth_frequency_comparison_df,
    "Depth + Frequency Ablation: Global Accuracy Comparison"
)


plot_fake_accuracy_by_transformation(
    depth_frequency_comparison_df,
    "Depth + Frequency Ablation: Real/Fake Accuracy by Transformation Type"
)

#Grad-CAM
Since RGB + Depth was more promising than RGB + Depth + Edge, we applied Grad-CAM to this model. Almost same reasoning is done with models with depth and frequency. The non-gated model was more promising so we applied Grad-CAM on it. Grad-CAM helps visualize the image regions that most influence the prediction, providing a qualitative interpretation of the model behavior.

##RGB + DEPTH visualization

In [ ]:
!python src/gradCam_depth.py \
  --csv_path {TEST_CSV} \
  --image_root {IMAGE_ROOT} \
  --depth_root {DEPTH_ROOT} \
  --checkpoint checkpoints/best_depth_edge_uncertainty.pt \
  --output_dir results/gradcam_depth \
  --task fake \
  --target predicted \
  --max_images 1 \
  --image_size {IMAGE_SIZE} \
  --num_workers {NUM_WORKERS} \
  --fake_filter 0 \
  --shuffle

In [ ]:
!python src/gradCam_depth.py \
  --csv_path {TEST_CSV} \
  --image_root {IMAGE_ROOT} \
  --depth_root {DEPTH_ROOT} \
  --checkpoint checkpoints/best_depth_edge_uncertainty.pt \
  --output_dir results/gradcam_depth \
  --task fake \
  --target predicted \
  --max_images 1 \
  --image_size {IMAGE_SIZE} \
  --num_workers {NUM_WORKERS} \
  --fake_filter 1 \
  --shuffle

In [ ]:
!python src/gradCam_depth.py \
  --csv_path {TEST_CSV} \
  --image_root {IMAGE_ROOT} \
  --depth_root {DEPTH_ROOT} \
  --checkpoint checkpoints/best_depth_edge_uncertainty.pt \
  --output_dir results/gradcam_depth \
  --task transform \
  --target predicted \
  --max_images 1 \
  --image_size {IMAGE_SIZE} \
  --num_workers {NUM_WORKERS} \
  --transform_filter 0 \
  --shuffle

In [ ]:
!python src/gradCam_depth.py \
  --csv_path {TEST_CSV} \
  --image_root {IMAGE_ROOT} \
  --depth_root {DEPTH_ROOT} \
  --checkpoint checkpoints/best_depth_edge_uncertainty.pt \
  --output_dir results/gradcam_depth \
  --task transform \
  --target predicted \
  --max_images 1 \
  --image_size {IMAGE_SIZE} \
  --num_workers {NUM_WORKERS} \
  --transform_filter 1 \
  --shuffle

In [ ]:
!python src/gradCam_depth.py \
  --csv_path {TEST_CSV} \
  --image_root {IMAGE_ROOT} \
  --depth_root {DEPTH_ROOT} \
  --checkpoint checkpoints/best_depth_edge_uncertainty.pt \
  --output_dir results/gradcam_depth \
  --task transform \
  --target predicted \
  --max_images 1 \
  --image_size {IMAGE_SIZE} \
  --num_workers {NUM_WORKERS} \
  --transform_filter 2 \
  --shuffle

In [ ]:
gradcam_dir = Path("results/gradcam_depth")

for img_path in sorted(gradcam_dir.glob("*.png")):
    print(img_path.name)
    display(Image(filename=str(img_path)))

##RGB + DEPTH + FREQUENCY visualization

In [ ]:
!python src/gradcam_depth_frequency_gated.py \
  --csv_path {TEST_CSV} \
  --image_root {IMAGE_ROOT} \
  --depth_root {DEPTH_ROOT} \
  --checkpoint checkpoints/best_depth_frequency_gated_uncertainty.pt \
  --output_dir results/gradcam_depth_frequency \
  --task fake \
  --target predicted \
  --max_images 1 \
  --image_size {IMAGE_SIZE} \
  --num_workers {NUM_WORKERS} \
  --fake_filter 0 \
  --shuffle


In [ ]:
!python src/gradcam_depth_frequency_gated.py \
  --csv_path {TEST_CSV} \
  --image_root {IMAGE_ROOT} \
  --depth_root {DEPTH_ROOT} \
  --checkpoint checkpoints/best_depth_frequency_gated_uncertainty.pt \
  --output_dir results/gradcam_depth_frequency \
  --task fake \
  --target predicted \
  --max_images 1 \
  --image_size {IMAGE_SIZE} \
  --num_workers {NUM_WORKERS} \
  --fake_filter 1 \
  --shuffle


In [ ]:
!python src/gradcam_depth_frequency_gated.py \
  --csv_path {TEST_CSV} \
  --image_root {IMAGE_ROOT} \
  --depth_root {DEPTH_ROOT} \
  --checkpoint checkpoints/best_depth_frequency_gated_uncertainty.pt \
  --output_dir results/gradcam_depth_frequency \
  --task transform \
  --target predicted \
  --max_images 1 \
  --image_size {IMAGE_SIZE} \
  --num_workers {NUM_WORKERS} \
  --transform_filter 0 \
  --shuffle



In [ ]:
!python src/gradcam_depth_frequency_gated.py \
  --csv_path {TEST_CSV} \
  --image_root {IMAGE_ROOT} \
  --depth_root {DEPTH_ROOT} \
  --checkpoint checkpoints/best_depth_frequency_gated_uncertainty.pt \
  --output_dir results/gradcam_depth_frequency \
  --task transform \
  --target predicted \
  --max_images 1 \
  --image_size {IMAGE_SIZE} \
  --num_workers {NUM_WORKERS} \
  --transform_filter 1 \
  --shuffle



In [ ]:
!python src/gradcam_depth_frequency_gated.py \
  --csv_path {TEST_CSV} \
  --image_root {IMAGE_ROOT} \
  --depth_root {DEPTH_ROOT} \
  --checkpoint checkpoints/best_depth_frequency_gated_uncertainty.pt \
  --output_dir results/gradcam_depth_frequency \
  --task transform \
  --target predicted \
  --max_images 1 \
  --image_size {IMAGE_SIZE} \
  --num_workers {NUM_WORKERS} \
  --transform_filter 2 \
  --shuffle

In [ ]:
gradcam_dir = Path("results/gradcam_depth_frequency")

for img_path in sorted(gradcam_dir.glob("*.png")):
    print(img_path.name)
    display(Image(filename=str(img_path)))

#ViT
We try to use different method for training to evaluate if there are some improvements.
In particular we decide to apply ViT for the RGB baseline (lamda_fake=1.0 lamda_transform=1.0), for the frequency and for frequency-rgb-depth.

##ViT RGB

**Train**

In [ ]:
!python src/train_vit_RGB_1_2.py \
  --train_csv  {TRAIN_CSV} \
  --val_csv    {VAL_CSV} \
  --image_root {IMAGE_ROOT} \
  --epochs     {EPOCHS} \
  --batch_size 16 \
  --num_workers {NUM_WORKERS} \
  --checkpoint_name best_vit_rgb_multitask_1_2.pt

**Evalutation**

In [ ]:
!python src/evaluate_vit_RGB.py \
  --csv_path   {TEST_CSV} \
  --image_root {IMAGE_ROOT} \
  --checkpoint checkpoints/best_vit_rgb_multitask_1_2.pt \
  --output_dir results/vit_rgb_multitask_1_2.pt\
  --batch_size 16 \
  --image_size {IMAGE_SIZE} \
  --num_workers {NUM_WORKERS}

**Results**

In [ ]:
show_evaluation_outputs("results/vit_multitask_1_1")

##ViT RGBs+Frequency+Depth

**PreTrain**

In [ ]:
def filter_csv_by_depth(csv_path, depth_root, output_csv):
    df         = pd.read_csv(csv_path)
    depth_root = Path(depth_root)

    mask = df["image_path"].apply(
        lambda p: (depth_root / Path(p).with_suffix(".npy")).exists()
    )

    df_filtered = df[mask].reset_index(drop=True)
    df_filtered.to_csv(output_csv, index=False)

    print(f"[{Path(csv_path).name}]")
    print(f"  Original: {len(df)} rows")
    print(f"  Filtred:  {len(df_filtered)} rows")
    print(f"  Removed:   {len(df) - len(df_filtered)} without depth map")
    print()
    return df_filtered


filter_csv_by_depth(TRAIN_CSV, DEPTH_ROOT, "data/splits/train_depth.csv")
filter_csv_by_depth(VAL_CSV,   DEPTH_ROOT, "data/splits/val_depth.csv")
filter_csv_by_depth(TEST_CSV,  DEPTH_ROOT, "data/splits/test_depth.csv")

**ReName**

In [ ]:
TRAIN_CSV_DEPTH = "data/splits/train_depth.csv"
VAL_CSV_DEPTH   = "data/splits/val_depth.csv"
TEST_CSV_DEPTH  = "data/splits/test_depth.csv"

**Train**

In [ ]:
!python src/train_vit_depth_frequency_1_2.py \
  --train_csv  {TRAIN_CSV_DEPTH} \
  --val_csv    {VAL_CSV_DEPTH} \
  --image_root {IMAGE_ROOT} \
  --depth_root {DEPTH_ROOT} \
  --epochs     {EPOCHS} \
  --batch_size 16 \
  --num_workers 0 \
  --checkpoint_name best_vit_depth_freq_multitask_1_2.pt

**Evaluate**

In [ ]:
!python src/evaluate_vit_depth_frequency.py \
  --csv_path   {TEST_CSV} \
  --image_root {IMAGE_ROOT} \
  --depth_root {DEPTH_ROOT} \
  --checkpoint checkpoints/best_vit_depth_freq_multitask_1_2.pt \
  --output_dir results/vit_depth_freq_multitask_1_2 \
  --batch_size 16 --num_workers {NUM_WORKERS}

**Results**

In [ ]:
show_evaluation_outputs("results/vit_depth_freq_multitask_1_2")

##Comparison

In [ ]:
RESNET_RESULTS = {
    "ResNet fake-only":         "results/rgb_fake/metrics.json",
    "ResNet transform-only":    "results/rgb_transform/metrics.json",
    "ResNet multitask 1-1":     "results/rgb_multitask_1_1/metrics.json",
    "ResNet multitask 1-2":     "results/rgb_multitask_1_2/metrics.json",
    "ResNet multitask 2-1":     "results/rgb_multitask_2_1/metrics.json",
    "ResNet multitask learned": "results/rgb_multitask_learned_weights/metrics.json",
    "ResNet D+F 1-1":           "results/depth_frequency_1_1/metrics.json",
    "ResNet D+F 1-2":           "results/depth_frequency_1_2/metrics.json",
}

VIT_RESULTS = {
    "ViT multitask 1-1":        "results/vit_rgb_multitask_1_1/metrics.json",
    "ViT multitask 1-2":        "results/vit_rgb_multitask_1_2/metrics.json",
    "ViT multitask learned":    "results/vit_rgb_multitask_learned/metrics.json",
    "ViT D+F 1-1":              "results/vit_depth_freq_multitask_1_1/metrics.json",
    "ViT D+F 1-2":              "results/vit_depth_freq_multitask_1_2/metrics.json",
}

compare_resnet_vs_vit(RESNET_RESULTS, VIT_RESULTS)